In [ ]:
import time as t
import tkinter as tk
class node:
    def __init__(self, andar,id=None):
        self.andar = andar
        self.proxE = None
        self.backE = None
        

class nodeF:
    def __init__(self, valor):
        self.valor = valor 
        self.prox = None

class filaEncadeada():
    def __init__(self):
        self.start = None
        self.fim = None

    def adicionar(self, atributo):
        novo = nodeF(atributo)
        if self.start is None and self.fim is None:
            self.start = novo
            self.fim = novo
        else:
            self.fim.prox = novo
            self.fim = novo
    
    def remover(self):
        aux = self.start
        if self.start is None:
            return None 
        elif self.start is self.fim:
            aux = self.start
            self.start = None
            self.fim = None
            return aux.valor
        else: 
            self.start = self.start.prox
            return aux.valor

class elevador:
    def __init__(self):
        self.head = None  
        self.end = None  
        self.atual = None

    def esta_vazia(self):
        return self.head is None
    
    def inserir(self, andar):
        novo_nodo = node(andar)
        if self.esta_vazia():
            self.head = novo_nodo
            self.end = novo_nodo
            self.atual = novo_nodo
        else:
            self.end.proxE = novo_nodo
            novo_nodo.backE = self.end
            self.end = novo_nodo

class elevadores:
    def __init__(self):
        self.conj = [] 
        
    def adicionar_elevadores(self, quantidade=3):
        for _ in range(quantidade):
            novo_elevador = elevador()
            self.conj.append(novo_elevador)
            
    def total_andares(self, total):
        for i in self.conj:
            for b in range(total):
                i.inserir(b)
                
    def maisProx(self, embarque):
        if not self.conj:
            print("Nenhum elevador cadastrado.")
            return None
            
        primeiro_elevador = self.conj[0]
        andar_min = primeiro_elevador.head.andar
        andar_max = primeiro_elevador.end.andar
        
        if embarque < andar_min or embarque > andar_max:
            print(f"Erro: O andar {embarque} não existe.")
            return None
            
        elevador_mais_proximo = None
        menor_distancia = float('inf')
        for elevador_atual in self.conj:
            distancia = abs(elevador_atual.atual.andar - embarque)
            if distancia < menor_distancia:
                menor_distancia = distancia
                elevador_mais_proximo = elevador_atual
                
        return elevador_mais_proximo


    def subirandar(self, destino, elevador_obj, fila):
        paradas = [destino] 
        
        while paradas and elevador_obj.atual.andar < max(paradas): 
            elevador_obj.atual = elevador_obj.atual.proxE

            #BACK END ONLY(maybe)
            print(f"   [↑] Elevador [{self.conj.index(elevador_obj)+1}] subindo... chegou ao andar {elevador_obj.atual.andar}")

            t.sleep(1.25)
            
            if elevador_obj.atual.andar in paradas:

                print(f"      -> [DESEMBARQUE] Passageiro no ELEVADOR [{self.conj.index(elevador_obj)+1}] chegou ao destino {elevador_obj.atual.andar}!")
                t.sleep(3)
                paradas = [p for p in paradas if p != elevador_obj.atual.andar]


            atual_fila = fila.start
            anterior_fila = None
            while atual_fila is not None:
                origem_carona, destino_carona = atual_fila.valor
                
                if origem_carona == elevador_obj.atual.andar and destino_carona > origem_carona:
                    print(f"      -> [CARONA] Passageiro embarcou no {origem_carona} rumo ao {destino_carona} com o elevador [{self.conj.index(elevador_obj)+1}]")
                    
                    t.sleep(4)
                    
                    paradas.append(destino_carona)
                    
                    if anterior_fila is None:
                        fila.start = atual_fila.prox
                        if fila.start is None: fila.fim = None
                    else:
                        anterior_fila.prox = atual_fila.prox
                        if atual_fila.prox is None: fila.fim = anterior_fila
                    
                    atual_fila = atual_fila.prox
                else:
                    anterior_fila = atual_fila
                    atual_fila = atual_fila.prox

    def descerandar(self, destino, elevador_obj, fila):
        paradas = [destino]
        
        while paradas and elevador_obj.atual.andar > min(paradas): 
            elevador_obj.atual = elevador_obj.atual.backE               
            print(f"   [↓] Elevador [{self.conj.index(elevador_obj)+1}] descendo... chegou ao andar {elevador_obj.atual.andar}")
            
            t.sleep(1.25)
            
            # 1. Checa desembarque
            if elevador_obj.atual.andar in paradas:
                print(f"      -> [DESEMBARQUE] Passageiro chegou ao destino {elevador_obj.atual.andar}!")
                
                t.sleep(4) #DELAY, PORTAS ABRINDO
                
                paradas = [p for p in paradas if p != elevador_obj.atual.andar]

            atual_fila = fila.start
            anterior_fila = None
            while atual_fila is not None:
                origem_carona, destino_carona = atual_fila.valor

                if origem_carona == elevador_obj.atual.andar and destino_carona < origem_carona:
                    print(f"      -> [CARONA] Passageiro embarcou no {origem_carona} rumo ao {destino_carona} utilizando o elevador {self.conj.index(elevador_obj)+1}!")
                    
                    t.sleep(4)
                    
                    paradas.append(destino_carona)
                    
                    # Remove o passageiro da fila
                    if anterior_fila is None:
                        fila.start = atual_fila.prox
                        if fila.start is None: fila.fim = None
                    else:
                        anterior_fila.prox = atual_fila.prox
                        if atual_fila.prox is None: fila.fim = anterior_fila
                    
                    atual_fila = atual_fila.prox
                else:
                    anterior_fila = atual_fila
                    atual_fila = atual_fila.prox
    
    def sobedesce(self, destino, e, fila):
        elevador_obj = self.maisProx(e)
        if elevador_obj is not None:
            while elevador_obj.atual.andar != e:
                if elevador_obj.atual.andar < e:
                    elevador_obj.atual = elevador_obj.atual.proxE
                else:
                    elevador_obj.atual = elevador_obj.atual.backE
            t.sleep(1) # porta abrindo       
            print(f"\n[INÍCIO] Elevador [{self.conj.index(elevador_obj)+1}] parou no andar {e} para o embarque principal.") 


            if elevador_obj.atual.andar == destino:
                print('Você já está no andar desejado')
            elif elevador_obj.atual.andar < destino:
                self.subirandar(destino, elevador_obj, fila)
            else: 
                self.descerandar(destino, elevador_obj, fila)


    def processar_fila(self, fila):
        while fila.start is not None:
            chamada = fila.remover()
            if chamada:
                origem, destino = chamada
                self.sobedesce(destino, origem, fila)



if __name__ == "__main__":
    predio = elevadores()
    predio.adicionar_elevadores() 
    predio.total_andares(10)       # Andares de 0 a 9
    
    fila_chamadas = filaEncadeada()

    fila_chamadas.adicionar((1, 8))
    fila_chamadas.adicionar((1, 5))
    fila_chamadas.adicionar((6, 9))
    fila_chamadas.adicionar((2, 0))
    fila_chamadas.adicionar((7, 0))
    fila_chamadas.adicionar((5, 2))
    fila_chamadas.adicionar((4, 8))

    
    predio.processar_fila(fila_chamadas)

    


[INÍCIO] Elevador [1] parou no andar 1 para o embarque principal.
   [↑] Elevador [1] subindo... chegou ao andar 2
   [↑] Elevador [1] subindo... chegou ao andar 3
   [↑] Elevador [1] subindo... chegou ao andar 4
      -> [CARONA] Passageiro embarcou no 4 rumo ao 8 com o elevador [1]


KeyboardInterrupt: 

In [2]:
import tkinter as tk
from tkinter import font

# 1. Cria a janela principal
janela = tk.Tk()
janela.title("SIMULADOR DE ELEVADORES")
janela.geometry("1200x900")

F=font.Font(family='Times New Roman', size=16,weight='bold')
iniciar = tk.Button(janela,text='iniciar simulação',font=F,width=17,height=5,background='#0000FF')
sair=tk.Button(janela,text='Sair',font=F,width=16,height=5,background='#0066FF',command=lambda:janela.destroy())
#sair.place(x=50,y=100)
sair.grid(row=0,column=1,padx=900,pady=700)
iniciar.place(x=1650,y=200)
#sair.pack(pady=200)

# 3. Mantém a janela aberta
janela.mainloop()

In [3]:
import tkinter as tk

# Configuração da Janela Principal
janela = tk.Tk()
janela.title("Simulador de Elevador")
janela.geometry("500x600")

# 1. CRIAR O CANVAS
# Definimos a largura (width), altura (height) e a cor de fundo (bg)
canvas = tk.Canvas(janela, width=200, height=500, bg="#1e1e1e", highlightthickness=0)
canvas.place(x=50, y=50)

# 2. DESENHAR OS ELEMENTOS VISUAIS
# Desenha as linhas do poço do elevador (X1, Y1, X2, Y2)
canvas.create_line(20, 10, 20, 490, fill="#555555", width=2)
canvas.create_line(180, 10, 180, 490, fill="#555555", width=2)

# Desenha a cabine do elevador (retângulo: X1, Y1, X2, Y2)
# Guardamos o ID do desenho na variável 'elevador_id' para conseguirmos movê-lo depois
elevador_id = canvas.create_rectangle(40, 400, 160, 480, fill="#007acc", outline="white", width=2)


# 3. CONECTAR COM O SEU BACK-END (Funções de Movimento)
def mover_subir():
    # O método .move(id, dx, dy) desloca o objeto. 
    # Em computação, Y diminui para SUBIR e aumenta para DESCER.
    canvas.move(elevador_id, 0, -10) 

def mover_descer():
    canvas.move(elevador_id, 0, 10)


# Painel de controle de testes lado esquerdo/direito
botao_subir = tk.Button(janela, text="▲ Subir", font=("Arial", 11), command=mover_subir)
botao_subir.place(x=280, y=200, width=100, height=40)

botao_descer = tk.Button(janela, text="▼ Descer", font=("Arial", 11), command=mover_descer)
botao_descer.place(x=280, y=260, width=100, height=40)

janela.mainloop()

In [ ]:
import time
import threading
import tkinter as tk
from tkinter import messagebox

# ---------------------------------------------------------
# ESTRUTURAS DE DADOS (Listas Encadeadas)
# ---------------------------------------------------------
class node:
    def __init__(self, andar):
        self.andar = andar
        self.proxE = None
        self.backE = None

class nodeF:
    def __init__(self, valor):
        self.valor = valor 
        self.prox = None

class filaEncadeada():
    def __init__(self):
        self.start = None
        self.fim = None
        # Trava de segurança para ambientes com múltiplas threads
        self.lock = threading.Lock() 

    def adicionar(self, atributo):
        with self.lock:
            novo = nodeF(atributo)
            if self.start is None and self.fim is None:
                self.start = novo
                self.fim = novo
            else:
                self.fim.prox = novo
                self.fim = novo
    
    def remover(self):
        with self.lock:
            aux = self.start
            if self.start is None:
                return None 
            elif self.start is self.fim:
                self.start = None
                self.fim = None
                return aux.valor
            else: 
                self.start = self.start.prox
                return aux.valor
            
    def esta_vazia(self):
        with self.lock:
            return self.start is None


# ---------------------------------------------------------
# LÓGICA DO ELEVADOR (Execução em Paralelo)
# ---------------------------------------------------------
class elevador(threading.Thread):
    def __init__(self, id_elevador):
        super().__init__(daemon=True) # Daemon garante que a thread feche ao fechar o app
        self.id_elevador = id_elevador
        self.head = None  
        self.end = None  
        self.atual = None
        self.chamada_atual = None
        self.status = "Parado"

    def inserir(self, andar):
        novo_nodo = node(andar)
        if self.head is None:
            self.head = novo_nodo
            self.end = novo_nodo
            self.atual = novo_nodo
        else:
            self.end.proxE = novo_nodo
            novo_nodo.backE = self.end
            self.end = novo_nodo

    def run(self):
        # Loop contínuo da thread do elevador
        while True:
            if self.chamada_atual:
                origem, destino = self.chamada_atual
                self.mover(origem, destino)
                self.chamada_atual = None
                self.status = "Parado"
            time.sleep(0.5)

    def mover(self, origem, destino):
        # 1. Vai até a origem buscar o passageiro
        self.status = f"Buscando no {origem}º"
        while self.atual.andar != origem:
            if self.atual.andar < origem:
                self.atual = self.atual.proxE
            else:
                self.atual = self.atual.backE
            time.sleep(0.8) # Tempo de deslocamento entre andares
        
        self.status = "Embarcando..."
        time.sleep(1.5) 
        
        # 2. Leva o passageiro ao destino
        self.status = f"Levando ao {destino}º"
        while self.atual.andar != destino:
            if self.atual.andar < destino:
                self.atual = self.atual.proxE
            else:
                self.atual = self.atual.backE
            time.sleep(0.8)
            
        self.status = "Desembarcando..."
        time.sleep(1.5)


class elevadores(threading.Thread):
    def __init__(self, fila_global, total_andares=10):
        super().__init__(daemon=True)
        self.conj = [] 
        self.fila = fila_global
        self.total_andares = total_andares
        
    def adicionar_elevadores(self, quantidade=3):
        for i in range(quantidade):
            novo_elevador = elevador(i+1)
            # Popula a lista encadeada de andares para este elevador
            for b in range(self.total_andares):
                novo_elevador.inserir(b)
            
            self.conj.append(novo_elevador)
            novo_elevador.start() # Inicia a thread isolada deste elevador
            
    def run(self):
        # Esta thread atua como "Despachante": lê a fila e atribui tarefas
        while True:
            if not self.fila.esta_vazia():
                # Filtra apenas elevadores parados (sem chamada atual)
                disponiveis = [e for e in self.conj if e.chamada_atual is None]
                
                if disponiveis:
                    chamada = self.fila.remover()
                    if chamada:
                        origem, destino = chamada
                        # Encontra o elevador disponível mais próximo da origem
                        mais_prox = min(disponiveis, key=lambda e: abs(e.atual.andar - origem))
                        mais_prox.chamada_atual = (origem, destino)
            
            time.sleep(0.5)


# ---------------------------------------------------------
# FRONTEND (Tkinter UI)
# ---------------------------------------------------------
class AppElevadores:
    def __init__(self, root, predio_manager):
        self.root = root
        self.predio = predio_manager
        self.root.title("Simulador de Elevadores Paralelos")
        self.root.geometry("600x650")
        
        # Inicia a thread despachante
        self.predio.start()

        # Adiciona chamadas iniciais automáticas (como no seu script original)
        self.predio.fila.adicionar((1, 8))
        self.predio.fila.adicionar((2, 0))
        self.predio.fila.adicionar((6, 9))

        self.montar_interface()
        self.atualizar_graficos()

    def montar_interface(self):
        frame_top = tk.Frame(self.root)
        frame_top.pack(pady=10)

        tk.Label(frame_top, text="Origem:").grid(row=0, column=0, padx=5)
        self.origem_entry = tk.Entry(frame_top, width=5)
        self.origem_entry.grid(row=0, column=1)

        tk.Label(frame_top, text="Destino:").grid(row=0, column=2, padx=5)
        self.destino_entry = tk.Entry(frame_top, width=5)
        self.destino_entry.grid(row=0, column=3)

        tk.Button(frame_top, text="Adicionar Chamada", command=self.adicionar_chamada, bg="lightblue").grid(row=0, column=4, padx=15)
        tk.Button(frame_top, text="Sair do Programa", command=self.root.quit, bg="salmon").grid(row=0, column=5)

        # Canvas onde o prédio será desenhado
        self.canvas_h = 500
        self.canvas_w = 500
        self.canvas = tk.Canvas(self.root, width=self.canvas_w, height=self.canvas_h, bg="white", highlightthickness=1, highlightbackground="black")
        self.canvas.pack(pady=10)

    def adicionar_chamada(self):
        try:
            o = int(self.origem_entry.get())
            d = int(self.destino_entry.get())
            
            if 0 <= o < self.predio.total_andares and 0 <= d < self.predio.total_andares:
                self.predio.fila.adicionar((o, d))
                self.origem_entry.delete(0, tk.END)
                self.destino_entry.delete(0, tk.END)
            else:
                messagebox.showwarning("Erro de Andar", f"Os andares vão de 0 a {self.predio.total_andares - 1}.")
        except ValueError:
            messagebox.showerror("Erro de Digitação", "Por favor, digite números inteiros válidos.")

    def atualizar_graficos(self):
        self.canvas.delete("all")
        
        # 1. Desenha as linhas horizontais (Andares)
        altura_andar = self.canvas_h / self.predio.total_andares
        for i in range(self.predio.total_andares):
            y = self.canvas_h - (i * altura_andar)
            self.canvas.create_line(0, y, self.canvas_w, y, fill="lightgray", dash=(4, 2))
            self.canvas.create_text(25, y - altura_andar/2, text=f"{i}º", font=("Arial", 12, "bold"))

        # 2. Desenha os elevadores
        largura_elevador = 50
        offset_x = 100
        espacamento = 120
        
        for idx, el in enumerate(self.predio.conj):
            if el.atual:
                andar_atual = el.atual.andar
                x0 = offset_x + (idx * espacamento)
                y1 = self.canvas_h - (andar_atual * altura_andar)
                y0 = y1 - altura_andar

                # Muda a cor visualmente para mostrar movimento
                cor_fundo = "#4CAF50" if el.status == "Parado" else "#FF9800"
                
                self.canvas.create_rectangle(x0, y0 + 5, x0 + largura_elevador, y1, fill=cor_fundo, outline="black")
                self.canvas.create_text(x0 + largura_elevador/2, y0 + 20, text=f"E{idx+1}", fill="white", font=("Arial", 10, "bold"))
                
                # Exibe o status (Buscando, Levando, etc.)
                self.canvas.create_text(x0 + largura_elevador/2, y1 + 10, text=el.status, fill="blue", font=("Arial", 8))

        # Re-chama essa função a cada 50ms para criar a animação sem travar o Tkinter
        self.root.after(50, self.atualizar_graficos)


if __name__ == "__main__":
    # Inicializa a Fila e o Prédio
    fila_chamadas = filaEncadeada()
    predio = elevadores(fila_chamadas, total_andares=10)
    predio.adicionar_elevadores(quantidade=3)
    
    # Inicializa a Interface Gráfica
    root = tk.Tk()
    app = AppElevadores(root, predio)
    root.mainloop()

: 

In [ ]:
import tkinter as tk
from tkinter import messagebox

#NENHUMA MUDANÇA AQUI, TUDO CERTO

class node:
    def __init__(self, andar):
        self.andar = andar
        self.proxE = None
        self.backE = None

class nodeF:
    def __init__(self, valor):
        self.valor = valor 
        self.prox = None

class filaEncadeada():
    def __init__(self):
        self.start = None
        self.fim = None

    def adicionar(self, atributo):
        novo = nodeF(atributo)
        if self.start is None and self.fim is None:
            self.start = novo
            self.fim = novo
        else:
            self.fim.prox = novo
            self.fim = novo
    
    def remover(self):
        aux = self.start
        if self.start is None:
            return None 
        elif self.start is self.fim:
            self.start = None
            self.fim = None
            return aux.valor
        else: 
            self.start = self.start.prox
            return aux.valor
            
    def esta_vazia(self):
        return self.start is None

#####################################################
# LÓGICA DO ELEVADOR (Máquina de Estados)

class elevador:
    def __init__(self, id_elevador):
        self.id_elevador = id_elevador
        self.head = None  
        self.end = None  
        self.atual = None
        self.chamada_atual = None
        
        # Variáveis de Estado (substituem o time.sleep)
        self.estado = "PARADO" 
        self.ticks_espera = 0 

    def inserir(self, andar):
        novo_nodo = node(andar)
        if self.head is None:
            self.head = novo_nodo
            self.end = novo_nodo
            self.atual = novo_nodo
        else:
            self.end.proxE = novo_nodo
            novo_nodo.backE = self.end
            self.end = novo_nodo

    def processar_passo(self):
        """Esta função é chamada várias vezes por segundo pelo Tkinter"""
        # Se o elevador está esperando (portas abrindo/fechando), apenas diminui o tempo
        if self.ticks_espera > 0:
            self.ticks_espera -= 1
            return

        if self.chamada_atual is None:
            self.estado = "PARADO"
            return

        origem, destino = self.chamada_atual

        # MÁQUINA DE ESTADOS:
        if self.estado == "INDO_ORIGEM":
            if self.atual.andar != origem:
                # Dá um "passo" na lista encadeada
                if self.atual.andar < origem: self.atual = self.atual.proxE
                else: self.atual = self.atual.backE
            else:
                self.estado = "EMBARCANDO"
                self.ticks_espera = 3  # Espera 3 "ticks" para simular o embarque

        elif self.estado == "EMBARCANDO":
            self.estado = "INDO_DESTINO"

        elif self.estado == "INDO_DESTINO":
            if self.atual.andar != destino:
                if self.atual.andar < destino: self.atual = self.atual.proxE
                else: self.atual = self.atual.backE
            else:
                self.estado = "DESEMBARCANDO"
                self.ticks_espera = 3

        elif self.estado == "DESEMBARCANDO":
            self.estado = "PARADO"
            self.chamada_atual = None # Fim da viagem, libera o elevador


class despachante:
    def __init__(self, fila_global, total_andares=10):
        self.conj = [] 
        self.fila = fila_global
        self.total_andares = total_andares
        
    def adicionar_elevadores(self, quantidade=3):
        for i in range(quantidade):
            novo_elevador = elevador(i+1)
            for b in range(self.total_andares):
                novo_elevador.inserir(b)
            self.conj.append(novo_elevador)
            
    def gerenciar_predio(self):
        """Distribui as chamadas e faz os elevadores andarem 1 passo"""
        # 1. Tenta atribuir novas chamadas aos elevadores parados
        if not self.fila.esta_vazia():
            disponiveis = [e for e in self.conj if e.estado == "PARADO"]
            if disponiveis:
                chamada = self.fila.remover()
                if chamada:
                    origem, destino = chamada
                    mais_prox = min(disponiveis, key=lambda e: abs(e.atual.andar - origem))
                    mais_prox.chamada_atual = (origem, destino)
                    mais_prox.estado = "INDO_ORIGEM"

        # 2. Manda cada elevador executar sua ação do momento
        for e in self.conj:
            e.processar_passo()

# ---------------------------------------------------------
# FRONTEND (Tkinter UI)
# ---------------------------------------------------------
class AppElevadores:
    def __init__(self, root, despachante_predio):
        self.root = root
        self.predio = despachante_predio
        self.root.title("Simulador Event-Driven (Sem Threads)")
        self.root.geometry("600x650")

        # Adiciona chamadas iniciais
        self.predio.fila.adicionar((1, 8))
        self.predio.fila.adicionar((2, 0))
        self.predio.fila.adicionar((6, 9))

        self.montar_interface()
        
        # Inicia o "Game Loop" do Tkinter
        self.loop_principal()

    def montar_interface(self):
        frame_top = tk.Frame(self.root)
        frame_top.pack(pady=10)

        tk.Label(frame_top, text="Origem:").grid(row=0, column=0, padx=5)
        self.origem_entry = tk.Entry(frame_top, width=5)
        self.origem_entry.grid(row=0, column=1)

        tk.Label(frame_top, text="Destino:").grid(row=0, column=2, padx=5)
        self.destino_entry = tk.Entry(frame_top, width=5)
        self.destino_entry.grid(row=0, column=3)

        tk.Button(frame_top, text="Adicionar Chamada", command=self.adicionar_chamada, bg="lightblue").grid(row=0, column=4, padx=15)
        tk.Button(frame_top, text="Sair", command=lambda:self.root.destroy(), bg="salmon").grid(row=0, column=5)

        self.canvas_h = 500
        self.canvas_w = 500
        self.canvas = tk.Canvas(self.root, width=self.canvas_w, height=self.canvas_h, bg="white", highlightthickness=1, highlightbackground="black")
        self.canvas.pack(pady=10)

    def adicionar_chamada(self):
        try:
            o = int(self.origem_entry.get())
            d = int(self.destino_entry.get())
            if 0 <= o < self.predio.total_andares and 0 <= d < self.predio.total_andares:
                self.predio.fila.adicionar((o, d))
                self.origem_entry.delete(0, tk.END)
                self.destino_entry.delete(0, tk.END)
            else:
                messagebox.showwarning("Erro", f"Os andares vão de 0 a {self.predio.total_andares - 1}.")
        except ValueError:
            messagebox.showerror("Erro", "Digite inteiros válidos.")

    def loop_principal(self):
        """O Coração do programa. Executa a lógica e desenha, repetidamente."""
        # 1. Roda a lógica do prédio (1 passo)
        self.predio.gerenciar_predio()

        # 2. Desenha o novo estado na tela
        self.canvas.delete("all")
        altura_andar = self.canvas_h / self.predio.total_andares
        
        for i in range(self.predio.total_andares):
            y = self.canvas_h - (i * altura_andar)
            self.canvas.create_line(0, y, self.canvas_w, y, fill="lightgray", dash=(4, 2))
            self.canvas.create_text(25, y - altura_andar/2, text=f"{i}º", font=("Arial", 12, "bold"))

        largura_elevador = 50
        for idx, el in enumerate(self.predio.conj):
            if el.atual:
                andar_atual = el.atual.andar
                x0 = 100 + (idx * 120)
                y1 = self.canvas_h - (andar_atual * altura_andar)
                y0 = y1 - altura_andar

                cor_fundo = "#4CAF50" if el.estado == "PARADO" else "#FF9800"
                self.canvas.create_rectangle(x0, y0 + 5, x0 + largura_elevador, y1, fill=cor_fundo, outline="black")
                self.canvas.create_text(x0 + largura_elevador/2, y0 + 20, text=f"E{idx+1}", fill="white", font=("Arial", 10, "bold"))
                
                # Exibe o estado da máquina de estados
                self.canvas.create_text(x0 + largura_elevador/2, y1 + 10, text=el.estado, fill="blue", font=("Arial", 8))

        # 3. Agenda o próximo ciclo para daqui a 500 milissegundos
        self.root.after(1000, self.loop_principal)


if __name__ == "__main__":
    fila_chamadas = filaEncadeada()
    
    # Substituímos a thread por uma classe normal de despachante
    predio = despachante(fila_chamadas, total_andares=10)
    predio.adicionar_elevadores(quantidade=3)
    
    root = tk.Tk()
    app = AppElevadores(root, predio)
    root.mainloop()